<a href="https://colab.research.google.com/github/marcinsawinski/scikit-learn-mooc/blob/main/workbooks/m6_02_boosting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📒 Workbook M6.02 — Learning from mistakes

In the lab we met **boosting**: trees trained one after the other, each
one correcting the errors of those before it. In this workbook you build
a gradient boosting model **by hand**, three trees and a few lines of
code, on the wages of football players. You will then let scikit-learn
decide when to stop adding trees, and hand a column of text to a model
without encoding it yourself.

**How this workbook works**

- The code is written for you. You do not need to write code, but you need
  to **understand every step**.
- Every `____` is a gap. Fill it in with the value asked for in the text
  **above** the cell. The error `NameError: name '____' is not defined`
  means: a gap is still empty.
- `check(...)` tells you whether your value is right (✅) or stops with a
  hint (❌).
- `commit(...)` records a **prediction** before you see the result. Be
  honest: a wrong prediction costs nothing, it is how you learn.
- Questions marked **🧠 Think** are not checked here. You will need the
  answers for the Moodle quiz and for the discussion at the start of the
  next lab.

In [ ]:
#@title Setup — run this cell first (no need to read it)
import hashlib
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)

ANSWERS = {
    "binning_loses_information": "9390298f3fb0c5b1",
    "new_prediction": "c4fe6b6dbe94790f",
    "new_residual": "40510175845988f1",
    "residual_example": "a176eeb31e601c38",
    "second_tree_learns": "222a2b9109feabcc",
    "testing_error_after_100_trees": "c72dd4da7d67f549",
    "trains_in_parallel": "3722c1dac3da9d15",
    "trees_built": "c85cff3d172e97ad"
}

def _normalize(value):
    if isinstance(value, float) and value.is_integer():
        value = int(value)
    if isinstance(value, float):
        value = round(value, 2)
    if isinstance(value, (list, tuple, set)):
        return ",".join(sorted(str(v).strip().lower() for v in value))
    return str(value).strip().lower()


def _hash(value):
    return hashlib.sha256(_normalize(value).encode()).hexdigest()[:16]


def check(name, value, hint="Re-read the text above this cell and try again."):
    """Stop with a hint unless `value` is the expected answer for `name`."""
    if _hash(value) != ANSWERS[name]:
        raise ValueError(f"❌ {name} = {value!r} is not right. {hint}")
    print(f"✅ {name}")


def check_score(name, value, actual):
    """Stop with a hint unless `value` is `actual` rounded to 2 decimals.

    For results that differ slightly between computers (e.g. k-NN ties).
    """
    if not isinstance(value, (int, float)) or abs(value - round(actual, 2)) > 0.005:
        raise ValueError(
            f"❌ {name} = {value!r} is not right. "
            "Read the score printed above and round it to 2 decimals."
        )
    print(f"✅ {name}")


def commit(prediction, options):
    """Record a prediction before seeing the result."""
    if prediction not in options:
        raise ValueError(f"Choose one of {options} first.")
    print(f"📌 Your prediction: {prediction!r}. Now run the next cell and compare.")

## 1. Data and target

The 8000 players of workbook M6.01 (video game *FIFA 22*), and their
weekly wage `wage_eur` to predict. To see what boosting does, we start
with a single feature: the `overall` rating of the player, out of 100.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split

players = pd.read_csv(
    "https://raw.githubusercontent.com/marcinsawinski/scikit-learn-mooc/refs/heads/main/datasets/football_players.csv"
)
target = players["wage_eur"]
numerical = [name for name in players.columns if name not in ("wage_eur", "nationality_name")]
train, test = train_test_split(players, random_state=0)
target_train, target_test = train["wage_eur"], test["wage_eur"]

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.scatter(train["overall"], target_train, s=4, alpha=0.3)
_ = ax.set(xlabel="overall rating", ylabel="weekly wage (€)")

The wage hardly moves up to a rating of about 70 and then shoots up.

## 2. Gradient boosting by hand

The recipe has three ingredients.

1. Start with the simplest prediction there is: the **mean** wage, for
   everybody.
2. Compute what is still unexplained: the **residuals**, that is the
   true wage minus the current prediction.
3. Train a **small** tree to predict these residuals, and **add** its
   predictions to the current ones. Go back to step 2.

✏️ A player earns €10 000. The model currently predicts €7 000 for him.
What is his residual, the number the next tree will try to predict?

In [ ]:
residual_example = ____
check("residual_example", residual_example, hint="Residual = true value − current prediction.")

✏️ The next tree predicts €2 000 for this player and is added to the
model. What does the model predict for him now, and what is his **new**
residual?

In [ ]:
new_prediction = ____
new_residual = ____
check("new_prediction", new_prediction)
check("new_residual", new_residual)

Now on the real data, with three small trees (`max_depth=2`):

In [ ]:
from sklearn.tree import DecisionTreeRegressor

rating_train = train[["overall"]]
rating_axis = pd.DataFrame({"overall": np.arange(47, 94)})

prediction_train = np.full(len(train), target_train.mean())
prediction_axis = np.full(len(rating_axis), target_train.mean())
stages = [("start: the mean wage", prediction_axis.copy())]
hand_made_trees = []
for step in range(1, 4):
    residuals = target_train - prediction_train
    small_tree = DecisionTreeRegressor(max_depth=2, random_state=0).fit(rating_train, residuals)
    hand_made_trees.append(small_tree)
    prediction_train = prediction_train + small_tree.predict(rating_train)
    prediction_axis = prediction_axis + small_tree.predict(rating_axis)
    stages.append((f"after tree {step}", prediction_axis.copy()))

fig, axes = plt.subplots(1, 4, figsize=(15, 3.5), sharey=True)
for ax, (title, curve) in zip(axes, stages):
    ax.scatter(train["overall"], target_train, s=3, alpha=0.2)
    ax.plot(rating_axis["overall"], curve, color="black", linewidth=2)
    ax.set(title=title, xlabel="overall rating", ylim=(0, 150_000))
_ = axes[0].set_ylabel("weekly wage (€)")

Each tree is a crude staircase with at most four steps. Added together,
they bend towards the players they were getting most wrong: the stars.

✏️ What does the **second** tree learn to predict? Write `"the wages"`
or `"the errors left by the first tree"`.

In [ ]:
second_tree_learns = ____
check("second_tree_learns", second_tree_learns)

Is this really what scikit-learn's `GradientBoostingRegressor` does? We
ask it for the same three trees and compare its predictions with ours.

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

official = GradientBoostingRegressor(n_estimators=3, max_depth=2, learning_rate=1.0, random_state=0)
official.fit(rating_train, target_train)
largest_difference = np.abs(official.predict(rating_axis) - prediction_axis).max()
print(f"largest difference between our predictions and scikit-learn's: {largest_difference:.6f} €")

Identical. `learning_rate=1.0` means "add each tree in full", as we did.
A smaller learning rate adds only a fraction of each tree: more cautious
steps, and more trees needed. Workbook M6.03 looks at that trade-off.

### Bagging and boosting side by side

| | bagging / random forest | boosting |
|---|---|---|
| each tree is trained on | a bootstrap sample of the players | the errors of the trees before it |
| the trees are | deep, each one overfits | small, each one underfits |
| the predictions are | averaged | added up |
| the ensemble mainly reduces | variance | bias |

✏️ In which method can the trees be trained **at the same time**, on
several processors? Write `"bagging"` or `"boosting"`.

In [ ]:
trains_in_parallel = ____
check("trains_in_parallel", trains_in_parallel, hint="In which method does a tree need the result of the previous ones?")

## 3. When to stop adding trees?

Now with all 27 numerical features. Each new tree reduces the error on
the training players. On new players, there comes a point where more
trees stop helping. `staged_predict` gives the predictions of the model
after 1 tree, 2 trees, 3 trees… so one training is enough to draw the
whole curve.

In [ ]:
boosting = GradientBoostingRegressor(n_estimators=300, learning_rate=0.3, random_state=0)
boosting.fit(train[numerical], target_train)
train_curve = [mean_absolute_error(target_train, p) for p in boosting.staged_predict(train[numerical])]
test_curve = [mean_absolute_error(target_test, p) for p in boosting.staged_predict(test[numerical])]

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(range(1, 301), train_curve, label="training error")
ax.plot(range(1, 301), test_curve, label="testing error")
ax.set(xlabel="number of trees", ylabel="error (€ per week)", ylim=(2000, 8000))
ax.legend()
best_n_trees = int(np.argmin(test_curve)) + 1
print(f"lowest testing error: {min(test_curve):.0f} € with {best_n_trees} trees; with 300 trees: {test_curve[-1]:.0f} €")

✏️ Between 100 and 300 trees, the training error keeps going down. What
does the testing error do? Write `"keeps going down"` or `"stops
improving"`.

In [ ]:
testing_error_after_100_trees = ____
check("testing_error_after_100_trees", testing_error_after_100_trees)

Unlike a forest, a boosting model **can overfit by having too many
trees**: the late trees correct errors that are only noise in the
training players.

We read the best number of trees on the test set, which is cheating (the
test set then no longer measures anything). **Early stopping** does it
properly: the model sets aside part of the **training** data, watches
the error on it, and stops when it has not improved for a while.

🔮 **Predict first.** We allow up to 1000 trees and stop when the
validation error has not improved for 10 trees in a row. How many trees
will be built? **"almost 1000"**, **"about half"** or **"far fewer"**?

In [ ]:
my_prediction = ____
commit(my_prediction, ["almost 1000", "about half", "far fewer"])

In [ ]:
early = GradientBoostingRegressor(
    n_estimators=1000, learning_rate=0.3, n_iter_no_change=10, validation_fraction=0.2, random_state=0
)
early.fit(train[numerical], target_train)
early_error = mean_absolute_error(target_test, early.predict(test[numerical]))
print(f"trees allowed: 1000, trees built: {early.n_estimators_}, testing error {early_error:.0f} €")

✏️ How many of the 1000 allowed trees were built? Write `"almost all"`
or `"far fewer"`.

In [ ]:
trees_built = ____
check("trees_built", trees_built)

With early stopping, `n_estimators` is no longer a value to tune: it is
an upper limit, and the model finds the right number by itself. The
price is the part of the training data put aside to watch the error.

## 4. Histogram gradient boosting

`HistGradientBoostingRegressor` is scikit-learn's modern boosting model.
Before training it sorts the values of each feature into at most 255
**bins**. A tree then only tries the bin edges as thresholds, not every
value, which makes training much faster on large data.

In [ ]:
from time import perf_counter

from sklearn.ensemble import HistGradientBoostingRegressor

timings = {}
for name, model in [
    ("GradientBoostingRegressor", GradientBoostingRegressor(n_estimators=200, random_state=0)),
    ("HistGradientBoostingRegressor", HistGradientBoostingRegressor(max_iter=200, early_stopping=False, random_state=0)),
]:
    start = perf_counter()
    model.fit(train[numerical], target_train)
    timings[name] = {
        "training time (s)": perf_counter() - start,
        "testing error (€)": mean_absolute_error(target_test, model.predict(test[numerical])),
    }
pd.DataFrame(timings).T.round(1)

The times depend on your computer; the errors are close. On a table of
a million rows the difference in time is between minutes and hours.

✏️ The `overall` rating takes fewer than 50 different values. Does
sorting it into 255 bins lose any information? Write `"yes"` or `"no"`.

In [ ]:
binning_loses_information = ____
check("binning_loses_information", binning_loses_information, hint="Compare the number of different values with the number of bins.")

### ➕ Going further: a text column, without encoding it

So far we ignored `nationality_name`, a column with more than a hundred
countries. One-hot encoding it would create more than a hundred columns.
Histogram gradient boosting can use a categorical column **directly**:
a question can then be "is the player from one of **these** countries?".
All it needs is the column stored with pandas' `category` type.

In [ ]:
with_country = players[numerical + ["nationality_name"]].astype({"nationality_name": "category"})
train_country, test_country = with_country.loc[train.index], with_country.loc[test.index]
print(f"{with_country['nationality_name'].nunique()} countries")

native = HistGradientBoostingRegressor(categorical_features="from_dtype", random_state=0)
native.fit(train_country, target_train)
without = HistGradientBoostingRegressor(random_state=0).fit(train[numerical], target_train)
print(f"without the nationality: error {mean_absolute_error(target_test, without.predict(test[numerical])):.0f} €")
print(f"with the nationality:    error {mean_absolute_error(target_test, native.predict(test_country)):.0f} €")

No encoder, no pipeline, and the model also accepts missing values
without an imputer, like the tree of workbook M5.01. For tables that mix
numbers and categories, this is usually the first model worth trying.
scikit-learn's example
[Categorical feature support in gradient boosting](https://scikit-learn.org/1.8/auto_examples/ensemble/plot_gradient_boosting_categorical.html)
compares this with the encoders of module 1.

🧠 **Think.**

1. The gain from the nationality is small here. Does a player's country
   tell much about his wage once his ratings are known?
2. A linear model given the nationality as numbers 0, 1, 2… would treat
   them as an order (workbook M1.03). Why is that not a problem for the
   question "is the player from one of these countries?"

## 🎤 Before the next lab

1. Do the **Moodle quiz** for this workbook (mandatory, not graded).
2. At the start of the next lab, some of you will be asked to present
   (3 minutes, using your notebook on the projector) **one** of:
   - gradient boosting by hand: the four pictures and what each tree
     learns;
   - bagging against boosting: what the trees are trained on, how big
     they are, what the ensemble reduces;
   - why a boosting model can have too many trees, and how early
     stopping chooses their number;
   - what histogram gradient boosting changes, and how it uses a
     categorical column.